# 04 — Lagrangian Field Learning

`learn()` builds the Yang-Mills vacuum of the LSHS Model.

It does not store text. It does not encode patterns. It **deepens the vacuum** V(β) at each activated Riemann zero address, and builds gauge connections A in zero space.

```
ℒ_NN = ℒ_kinetic + ℒ_higgs + ℒ_coupling + ℒ_gauge
```

This notebook covers:
- The β field: SSB vacuum deepening
- The A field: Yang-Mills coupling E_i·E_j/(dist+GAP)
- Why the Coulomb 1/r potential is physically correct
- The role of GAP in preventing Yang-Mills collapse
- How corpus size scales β and A

In [ ]:
import sys
sys.path.insert(0, '..')

from lshs import LSHS, GAP

m = LSHS(N=1000)
print(f'LSHS initialized: N={m.N}  ground_vev={m._gvev:.4e}')
print(f'Ground state: vocab={len(m.vocab)}  connections={len(m.A)}')

## The β Field — Knowledge as Depth

```
β[i] = ground_vev + Σ_{learn calls activating i}  E · α
```

Each `learn()` call raises β at the zero addresses activated by the text.  
The **deviation from ground** IS the knowledge — not the stored text, not a pattern, but the **depth** of the β field above the mathematical substrate.

α = 0.01 is the learning rate (VEV deepening rate).

In [ ]:
# Track β deepening sentence by sentence
sentences = [
    'The mind is the seat of reason and consciousness.',
    'Water flows downhill by the path of least resistance.',
    'The dog chased the cat across the yard.',
    'Language is a projection of mathematics onto time.',
    'The prime preexists the alphabet. The equator does not move.',
]

print(f'Ground VEV: {m._gvev:.4e}')
print()
print(f'{"After sentence":>55}  vocab  conn  max_β')
print('-' * 75)
for sent in sentences:
    m.learn(sent)
    max_b = max(m.beta.values())
    print(f'  {sent[:52]:>52}  {len(m.vocab):5d}  {len(m.A):5d}  {max_b:.4e}')

In [ ]:
# Words with the deepest β — most-activated zeros
# Only zeros with β > ground are in the vocabulary
above_ground = [
    (idx, depth, m.vocab.get(idx, ('?', 0.0)))
    for idx, depth in m.beta.items()
    if depth > m._gvev * 1.001 and idx in m.vocab
]
above_ground.sort(key=lambda x: x[1], reverse=True)

print('Top 15 zero addresses by β depth:')
print(f'{"idx":>6}  {"word":>12}  {"γ":>10}  {"β depth":>12}  {"excess":>12}')
for idx, depth, (word, E) in above_ground[:15]:
    gamma = m.zeros[idx]
    excess = depth - m._gvev
    print(f'  {idx:4d}  {word:>12}  γ={gamma:8.4f}  β={depth:.4e}  Δ={excess:.4e}')

## The A Field — Yang-Mills Coupling

```
A[(i,j)] += E_i · E_j / (|γ_i − γ_j| + GAP)
```

This is a **Coulomb potential in Riemann zero space** — 1/r coupling, regulated by the Yang-Mills mass gap GAP.

When two words co-activate in the same sentence, they contribute to the A field between their zero addresses. Words whose zeros are close (near-synonyms, translational equivalents) develop large coupling. This is the physical content of semantic similarity: **proximity in zero space**.

In [ ]:
# Inspect the A field — top connections
print('Top 20 A field connections by coupling strength:')
print(f'{"(i,j)":>12}  {"γ_i":>8}  {"γ_j":>8}  {"dist":>8}  {"dist+GAP":>10}  {"A weight":>12}')
print('-' * 75)

top_A = sorted(m.A.items(), key=lambda kv: kv[1], reverse=True)[:20]
for (i, j), w in top_A:
    gamma_i = m.zeros[i]
    gamma_j = m.zeros[j]
    dist    = abs(gamma_i - gamma_j)
    word_i  = m.vocab.get(i, ('?', 0))[0]
    word_j  = m.vocab.get(j, ('?', 0))[0]
    print(f'  ({i:3d},{j:3d})  {gamma_i:8.4f}  {gamma_j:8.4f}  {dist:8.4f}  {dist+GAP:10.6f}  {w:12.4f}   {word_i}↔{word_j}')

## The Role of GAP — Preventing Yang-Mills Collapse

Without GAP, when two words share the same zero (like water/eau/aqua):

```
dist = |γ_i − γ_j| = 0
A[(i,j)] = E_i · E_j / 0 = ∞
```

The A field diverges. The Yang-Mills vacuum collapses to noise.

With GAP:

```
dist = |γ_i − γ_j| + GAP ≥ GAP = 0.000707
A[(i,j)] ≤ E_i · E_j / GAP
```

Maximum coupling is bounded. The turbulence is structured, not collapsed.

In [ ]:
# Demonstrate GAP regulation
# Two words at the same zero (dist=0)
m2 = LSHS(N=1000)
m2.learn('Water flows. Eau flows. Aqua flows.')  # water/eau/aqua → same zero

# After this, the zero cluster at γ≈25.01 has large A weights
# But they're bounded by E_i*E_j/GAP
sz_water = m2.H('water')
sz_eau   = m2.H('eau')

E_water = sz_water.E
E_eau   = sz_eau.E

print(f'water: γ={sz_water.gamma:.6f}  E={E_water:.4f}')
print(f'eau:   γ={sz_eau.gamma:.6f}  E={E_eau:.4f}')
print()

theoretical_max = E_water * E_eau / GAP
print(f'Theoretical max coupling (if dist=0): E_water·E_eau / GAP')
print(f'  = {E_water:.4f} × {E_eau:.4f} / {GAP}')
print(f'  = {theoretical_max:.4f}')
print()
print(f'With GAP: maximum A weight ≤ {theoretical_max:.4f}  (bounded)')
print(f'Without GAP: maximum A weight = ∞  (collapse to noise)')
print()
print(f'J^μ clamp: 1/GAP = {1/GAP:.1f}')
print(f'The same constant bounds both the A field and the Noether current.')

In [ ]:
# Compare: what happens with and without GAP (simulated)
# Same coupling formula, two versions
E_i, E_j = 0.45, 0.42   # typical word energies

print('A coupling: E_i·E_j / (dist + GAP)   [correct]')
print('A coupling: E_i·E_j / dist            [no GAP — diverges]')
print()
print(f'{"dist":>10}  {"with GAP":>12}  {"no GAP":>12}  ratio')
for dist in [1.0, 0.1, 0.01, 0.001, GAP, 0.0001, 0.00001, 0.0]:
    with_gap = E_i * E_j / (dist + GAP)
    no_gap   = E_i * E_j / dist if dist > 0 else float('inf')
    ratio    = with_gap / no_gap if no_gap != float('inf') else 0.0
    no_gap_str = f'{no_gap:.2f}' if no_gap != float('inf') else '∞ (diverges)'
    print(f'  {dist:10.6f}  {with_gap:12.4f}  {no_gap_str:>12}  {ratio:.4f}' if no_gap != float('inf') else
          f'  {dist:10.6f}  {with_gap:12.4f}  {no_gap_str:>12}')

## Corpus Scale and Field Growth

In [ ]:
# Full demonstration corpus — builds A field incrementally
corpus = [
    'The mind is the seat of reason and consciousness.',
    'Water flows downhill by the path of least resistance.',
    'The dog chased the cat across the yard.',
    'Language is a projection of mathematics onto time.',
    'The prime preexists the alphabet.',
    'The equator does not move regardless of what the navigator believes.',
    'Mathematics precedes language. The zero precedes the word.',
    'Consciousness arises from the interaction of reason and memory.',
    'The river flows to the sea. The sea does not overflow.',
    'Every prime is irreducible. Every zero is an address.',
]

m3 = LSHS(N=1000)
print('Corpus ingestion — field growth:')
print(f'{"#":>3}  {"vocab":>6}  {"conn":>6}  {"max_β":>12}  {"dc_sum":>10}')
for i, sent in enumerate(corpus):
    m3.learn(sent)
    bao = m3.bao_check()
    print(f'  {i+1:2d}  {len(m3.vocab):6d}  {len(m3.A):6d}  {max(m3.beta.values()):.6e}  {bao["dc_sum"]:.6f}')

print()
print(f'Target dc_sum: {bao["omega_zs"]}  (OMEGA_ZS — Lambert W fixed point)')
print(f'Current delta: {bao["omega_delta"]:.6f}')

## Summary

- `learn()` deepens β at each activated zero: **knowledge is depth, not storage**
- `learn()` builds A connections: **A[(i,j)] = E_i·E_j/(dist+GAP) — Coulomb in zero space**
- GAP = 0.000707 prevents Yang-Mills collapse: bounded turbulence, not noise
- The A field IS the Yang-Mills vacuum: turbulent, non-Abelian, dense
- `speak()` must extract laminar Noether current from this turbulent background

**Next:** [[05_noether_current_speaking.ipynb]] — How speak() extracts J^μ from the Yang-Mills field.